# Final Machine Learning Pipeline

Este notebook construye el pipeline **definitivo** para deploy producción. Las características principales del pipeline son:
- Usa sólo clases **open source**: `sklearn`, `feature-engine`.
- Incluye **custom transformers** para ingeniería de variables.
- Procesa únicamente las **features seleccionadas** tras feature selection.
- Incluye el **modelo entrenado** como último paso del pipeline (no sólo preprocesamiento).
- El **pipeline es un único objeto** serializable (`joblib`) capaz de transformar datos crudos y producir predicciones directamente (`pipeline.predict`).

---

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, Binarizer
from sklearn.base import BaseEstimator, TransformerMixin

from feature_engine.imputation import CategoricalImputer, MeanMedianImputer
from feature_engine.indicator import AddMissingIndicator
from feature_engine.transformation import LogTransformer, YeoJohnsonTransformer
from feature_engine.encoding import RareLabelEncoder, OrdinalEncoder
from feature_engine.selection import DropFeatures
from feature_engine.wrappers import SklearnTransformerWrapper

---

## 2. Cargar dataset y split train/test

In [ ]:
# Cargar el dataset principal
data = pd.read_csv('data/train.csv')

# Ejemplo de casteo: MSSubClass como string (ajustar según el dataset real)
if 'MSSubClass' in data.columns:
    data['MSSubClass'] = data['MSSubClass'].astype(str)

# Supongamos que es un problema de regresión con la variable target 'SalePrice'
TARGET = 'SalePrice'

# Separar X e y
y = data[TARGET]
X = data.drop(TARGET, axis=1)

# Split reproducible
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=0)

# Aplicar logaritmo natural al target para regresión
y_train = np.log(y_train)
y_test = np.log(y_test)

---

## 3. Configuración — Constantes

Las siguientes constantes **(en mayúsculas)** deben reflejar únicamente las features seleccionadas tras el feature selection.

In [ ]:
# Cargar lista de features seleccionadas del notebook 04
FEATURES = pd.read_csv('proyecto_pipeline_machine_learning_muestra/selected_features.csv', header=None)[0].tolist()

# Sólo las variables que aparecen en FEATURES
CATEGORICAL_VARS_WITH_NA_MISSING = [
    var for var in ['Alley','BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1',
                   'BsmtFinType2','FireplaceQu','GarageType','GarageFinish',
                   'GarageQual','GarageCond','PoolQC','Fence','MiscFeature']
    if var in FEATURES]

CATEGORICAL_VARS_WITH_NA_FREQUENT = [
    var for var in ['MasVnrType','Electrical']
    if var in FEATURES]

NUMERICAL_VARS_WITH_NA = [
    var for var in ['LotFrontage','GarageYrBlt','MasVnrArea']
    if var in FEATURES]

TEMPORAL_VARS = [
    var for var in ['YearRemodAdd','YrSold']
    if var in FEATURES]
REF_VAR = 'YearBuilt' if 'YearBuilt' in FEATURES else None

DROP_FEATURES = [var for var in ['Id'] if var in FEATURES]

NUMERICALS_LOG_VARS = [
    var for var in ['GrLivArea','1stFlrSF','TotalBsmtSF'] if var in FEATURES]

NUMERICALS_YEO_VARS = [var for var in ['LotArea'] if var in FEATURES]

BINARIZE_VARS = [var for var in ['ScreenPorch'] if var in FEATURES]

QUAL_VARS = [
    var for var in ['ExterQual','ExterCond','BsmtQual','BsmtCond','HeatingQC',
                   'KitchenQual','FireplaceQu','GarageQual','GarageCond','PoolQC']
    if var in FEATURES]

QUAL_MAPPINGS = {
    'Ex': 5,
    'Gd': 4,
    'TA': 3,
    'Fa': 2,
    'Po': 1,
    np.nan: 0
}

EXPOSURE_VARS = [var for var in ['BsmtExposure'] if var in FEATURES]
EXPOSURE_MAPPINGS = {
    'Gd': 4,
    'Av': 3,
    'Mn': 2,
    'No': 1,
    np.nan: 0
}

CATEGORICAL_VARS = [col for col in FEATURES if X_train[col].dtype=='O']

---

## 4. Reducir a features seleccionadas

In [ ]:
X_train = X_train[FEATURES].copy()
X_test = X_test[FEATURES].copy()

# Asegurar que las categóricas sean str
for col in CATEGORICAL_VARS:
    X_train[col] = X_train[col].astype(str)
    X_test[col] = X_test[col].astype(str)

---

## 5. Custom Transformers (si aplican)

In [ ]:
# Temporal variable transformer
class TemporalVariableTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, variables=None, reference=None):
        self.variables = variables
        self.reference = reference
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        for var in self.variables:
            X[var] = X[self.reference] - X[var]
        return X

# Mapper transformer (para variables cualitativas y exposición)
class Mapper(BaseEstimator, TransformerMixin):
    def __init__(self, variables, mappings):
        self.variables = variables
        self.mappings = mappings
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        for var in self.variables:
            X[var] = X[var].map(self.mappings)
        return X

---

## 6. Pipeline End-to-End con modelo entrenado

In [ ]:
# Armado del pipeline final
pipe_steps = []

# 1. Temporal
if TEMPORAL_VARS and REF_VAR:
    pipe_steps.append(('temporal', TemporalVariableTransformer(variables=TEMPORAL_VARS, reference=REF_VAR)))

# 2. Imputación numérica
if NUMERICAL_VARS_WITH_NA:
    pipe_steps.append((
        'num_imputer',
        MeanMedianImputer(imputation_method='median', variables=NUMERICAL_VARS_WITH_NA)
    ))

# 3. Indicador de nulos
if NUMERICAL_VARS_WITH_NA:
    pipe_steps.append((
        'add_na',
        AddMissingIndicator(variables=NUMERICAL_VARS_WITH_NA)
    ))

# 4. Imputación categórica NA = "Missing"
if CATEGORICAL_VARS_WITH_NA_MISSING:
    pipe_steps.append((
        'cat_imputer_missing',
        CategoricalImputer(imputation_method='missing', variables=CATEGORICAL_VARS_WITH_NA_MISSING)
    ))

# 5. Imputación categórica con moda
if CATEGORICAL_VARS_WITH_NA_FREQUENT:
    pipe_steps.append((
        'cat_imputer_frequent',
        CategoricalImputer(imputation_method='frequent', variables=CATEGORICAL_VARS_WITH_NA_FREQUENT)
    ))

# 6. Mapeo de cualitativos
if QUAL_VARS:
    pipe_steps.append(('qual_mapper', Mapper(variables=QUAL_VARS, mappings=QUAL_MAPPINGS)))

# 7. Mapeo de exposición
if EXPOSURE_VARS:
    pipe_steps.append(('exposure_mapper', Mapper(variables=EXPOSURE_VARS, mappings=EXPOSURE_MAPPINGS)))

# 8. Rare label encoding
rare_vars = [var for var in CATEGORICAL_VARS if var not in (QUAL_VARS + EXPOSURE_VARS)]
if rare_vars:
    pipe_steps.append((
        'rarelabel',
        RareLabelEncoder(tol=0.01, n_categories=1, variables=rare_vars)
    ))

# 9. Ordinal encoding
if rare_vars:
    pipe_steps.append((
        'ordinal',
        OrdinalEncoder(encoding_method='arbitrary', variables=rare_vars)
    ))

# 10. Log-transform
if NUMERICALS_LOG_VARS:
    pipe_steps.append((
        'log',
        LogTransformer(variables=NUMERICALS_LOG_VARS)
    ))

# 11. Yeo-Johnson transform
if NUMERICALS_YEO_VARS:
    pipe_steps.append((
        'yeo',
        YeoJohnsonTransformer(variables=NUMERICALS_YEO_VARS)
    ))

# 12. Binarizer
if BINARIZE_VARS:
    pipe_steps.append((
        'binarizer',
        SklearnTransformerWrapper(
            transformer=Binarizer(threshold=0), variables=BINARIZE_VARS
        )
    ))

# 13. Drop unused features
if DROP_FEATURES:
    pipe_steps.append((
        'drop_features',
        DropFeatures(features_to_drop=DROP_FEATURES)
    ))

# 14. Scaler
pipe_steps.append(('scaler', MinMaxScaler()))

# 15. Modelo final
pipe_steps.append(('lasso', Lasso(alpha=0.001, random_state=0)))

# Pipeline completo
price_pipe = Pipeline(steps=pipe_steps)

---

## 7. Entrenar pipeline

In [ ]:
# Ajuste SOLO sobre train
price_pipe.fit(X_train, y_train)

---

## 8. Evaluación del pipeline final

In [ ]:
# Predicciones en train y test
train_pred = price_pipe.predict(X_train)
test_pred = price_pipe.predict(X_test)

# Volver a la escala original del target
y_train_exp = np.exp(y_train)
y_test_exp = np.exp(y_test)
train_pred_exp = np.exp(train_pred)
test_pred_exp = np.exp(test_pred)

# Métricas
mse_train = mean_squared_error(y_train_exp, train_pred_exp)
mse_test = mean_squared_error(y_test_exp, test_pred_exp)
rmse_train = np.sqrt(mse_train)
rmse_test = np.sqrt(mse_test)
r2_train = r2_score(y_train_exp, train_pred_exp)
r2_test = r2_score(y_test_exp, test_pred_exp)

print('Train MSE:', mse_train)
print('Test  MSE:', mse_test)
print('Train RMSE:', rmse_train)
print('Test  RMSE:', rmse_test)
print('Train R2:', r2_train)
print('Test  R2:', r2_test)

Los resultados de desempeño aquí deben ser **idénticos** a los del notebook 06 dado que el pipeline encapsula TODO el preprocesamiento y modelización final.

In [ ]:
# Scatter predictions vs real
plt.figure(figsize=(8,6))
plt.scatter(y_train_exp, train_pred_exp, alpha=0.3, label='Train')
plt.scatter(y_test_exp, test_pred_exp, c='red', alpha=0.3, label='Test')
plt.xlabel('Real SalePrice')
plt.ylabel('Predicted SalePrice')
plt.title('Predicciones vs Valores Reales')
plt.legend()
plt.plot([y_train_exp.min(), y_train_exp.max()], [y_train_exp.min(), y_train_exp.max()], 'k--')
plt.show()

# Histograma de errores
train_errors = y_train_exp - train_pred_exp
test_errors = y_test_exp - test_pred_exp
plt.figure(figsize=(8,4))
plt.hist(train_errors, bins=30, alpha=0.6, label='Train')
plt.hist(test_errors, bins=30, alpha=0.6, label='Test', color='red')
plt.xlabel('Error')
plt.ylabel('Frecuencia')
plt.title('Distribución de Errores (Real - Predicción)')
plt.legend()
plt.show()

---

## 9. Guardar pipeline final con modelo

In [ ]:
joblib.dump(price_pipe, 'proyecto_pipeline_machine_learning_muestra/pipeline_completo.joblib')
print("Pipeline guardado exitosamente.")

---

## 10. Scoring sobre datos nuevos

In [ ]:
# Cargar nuevos datos (puede usar test.csv o cualquier otra fuente)
data_new = pd.read_csv('data/test.csv')

# Ejemplo de casteo
data_new['MSSubClass'] = data_new['MSSubClass'].astype(str)

# Reducir a features seleccionadas y mantener orden
X_new = data_new[FEATURES].copy()
for col in CATEGORICAL_VARS:
    if col in X_new:
        X_new[col] = X_new[col].astype(str)

# Detectar NaNs inesperados fuera de las imputadas
pipeline_na_handled = (
    set(NUMERICAL_VARS_WITH_NA) |
    set(CATEGORICAL_VARS_WITH_NA_MISSING) |
    set(CATEGORICAL_VARS_WITH_NA_FREQUENT)
)

# Columnas a controlar
unexpected_na_cols = [
    col for col in X_new.columns
    if (X_new[col].isnull().any() and col not in pipeline_na_handled)
]
if unexpected_na_cols:
    print(f"Dropeando filas con NaN inesperados en: {unexpected_na_cols}")
    orig_shape = X_new.shape
    X_new = X_new.dropna(subset=unexpected_na_cols)
    print(f"Filas iniciales: {orig_shape[0]}, filas tras drop: {X_new.shape[0]}")

> **Nota**: En producción, es fundamental dropear (o alertar) registros con NaNs en variables no contempladas por el pipeline, asegurando datos consistentes para scoring seguro.

In [ ]:
# Scoring final
y_new_hat = price_pipe.predict(X_new)
y_new_hat_exp = np.exp(y_new_hat)

plt.figure(figsize=(8,4))
plt.hist(y_new_hat_exp, bins=40, alpha=0.7, color='blue')
plt.xlabel('Predicted SalePrice')
plt.ylabel('Frecuencia')
plt.title('Histograma de Predicciones sobre Datos Nuevos')
plt.show()

Listo: **El pipeline completo está entrenado, probado, evaluado, guardado y demostrado haciendo scoring sobre datos nuevos.**